In [1]:
%pip install pymupdf sentence-transformers faiss-cpu pandas numpy tqdm requests python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import fitz
import re
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer

d:\WCE_Hack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        
        pages.append({
            "page_number": page_num + 1,
            "text": text
        })

    return pages

In [3]:
pages = extract_pdf_text("./data/Psychology2e_WEB.pdf")
print(len(pages))

753


In [4]:
import re

def split_into_sections(pages):
    section_data = []
    current_section = "unknown"

    for page in pages:
        text = page["text"]

        # Skip empty pages
        if not text or text.strip() == "":
            continue

        text = text.strip()

        # Detect section headers like:
        # 1.1 What Is Psychology?
        # 2.3 Neurons and Glial Cells
        matches = re.findall(r"\b\d+\.\d+\s+[A-Z][A-Za-z\s\-:]*", text)

        if matches:
            current_section = matches[0].strip()

        section_data.append({
            "section": current_section,
            "page_number": page["page_number"],
            "text": text
        })

    return section_data

In [5]:
section_pages = split_into_sections(pages)

print("Total section pages:", len(section_pages))
print(section_pages[0])
print(section_pages[1])

Total section pages: 751
{'section': 'unknown', 'page_number': 3, 'text': 'Psychology 2e \n \n \n \n \n \n \n \n \n \nSENIOR CONTRIBUTING AUTHORS \nROSE M. SPIELMAN, FORMERLY OF QUINNIPIAC UNIVERSITY \nWILLIAM J. JENKINS, MERCER UNIVERSITY \nMARILYN D. LOVETT, SPELMAN COLLEGE'}
{'section': '4.0 International License', 'page_number': 4, 'text': 'OpenStax \nRice University \n6100 Main Street MS-375 \nHouston, Texas 77005 \nTo learn more about OpenStax, visit https://openstax.org. \nIndividual print copies and bulk orders can be purchased through our website. \n©2020 Rice University. Textbook content produced by OpenStax is licensed under a Creative Commons \nAttribution 4.0 International License (CC BY 4.0). Under this license, any user of this textbook or the textbook \ncontents herein must provide proper attribution as follows:  \n-\nIf you redistribute this textbook in a digital format (including but not limited to PDF and HTML), then you\nmust retain on every page the following attri

In [6]:
for i in range(3):
    print("Page:", section_pages[i]["page_number"])
    print("Section:", section_pages[i]["section"])
    print(section_pages[i]["text"][:300])
    print("------")

Page: 3
Section: unknown
Psychology 2e 
 
 
 
 
 
 
 
 
 
SENIOR CONTRIBUTING AUTHORS 
ROSE M. SPIELMAN, FORMERLY OF QUINNIPIAC UNIVERSITY 
WILLIAM J. JENKINS, MERCER UNIVERSITY 
MARILYN D. LOVETT, SPELMAN COLLEGE
------
Page: 4
Section: 4.0 International License
OpenStax 
Rice University 
6100 Main Street MS-375 
Houston, Texas 77005 
To learn more about OpenStax, visit https://openstax.org. 
Individual print copies and bulk orders can be purchased through our website. 
©2020 Rice University. Textbook content produced by OpenStax is licensed under a Creativ
------
Page: 5
Section: 4.0 International License
OPENSTAX 
 
OpenStax provides free, peer-reviewed, openly licensed textbooks for introductory college and Advanced 
Placement® courses and low-cost, personalized courseware that helps students learn. A nonprofit ed tech 
initiative based at Rice University, we’re committed to helping students access
------


In [7]:
def chunk_text(section_chunks, chunk_size=400):
    final_chunks = []
    chunk_id = 0

    for item in section_chunks:
        text = item["text"]

        # Skip unwanted sections
        if any(x in text.lower() for x in [
            "review questions",
            "critical thinking questions",
            "personal application questions",
            "contents",
            "key terms"
        ]):
            continue

        words = text.split()

        for i in range(0, len(words), chunk_size):
            chunk = " ".join(words[i:i+chunk_size])

            if len(chunk.strip()) < 200:
                continue

            final_chunks.append({
                "chunk_id": chunk_id,
                "text": chunk,
                "section": item["section"],
                "page_number": item["page_number"]
            })
            chunk_id += 1

    return final_chunks

In [8]:
chunks = chunk_text(section_pages)

print("Total Chunks:", len(chunks))
print(chunks[0])

Total Chunks: 1076
{'chunk_id': 0, 'text': 'OPENSTAX OpenStax provides free, peer-reviewed, openly licensed textbooks for introductory college and Advanced Placement® courses and low-cost, personalized courseware that helps students learn. A nonprofit ed tech initiative based at Rice University, we’re committed to helping students access the tools they need to complete their courses and meet their educational goals. RICE UNIVERSITY OpenStax, OpenStax CNX, and OpenStax Tutor are initiatives of Rice University. As a leading research university with a distinctive commitment to undergraduate education, Rice University aspires to path-breaking research, unsurpassed teaching, and contributions to the betterment of our world. It seeks to fulfill this mission by cultivating a diverse community of learning and discovery that produces leaders across the spectrum of human endeavor. PHILANTHROPIC SUPPORT OpenStax is grateful for the generous philanthropic partners who advance our mission to improv

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 444.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
embedding = model.encode("What is psychology?")
print(len(embedding))

384


In [11]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

def build_faiss_index(chunks):
    texts = [c["text"] for c in chunks]

    embeddings = model.encode(texts, show_progress_bar=True)
    embeddings = np.array(embeddings)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index

In [12]:
index = build_faiss_index(chunks)
print("Index built successfully")

Batches: 100%|██████████| 34/34 [00:42<00:00,  1.24s/it]

Index built successfully


In [13]:
def retrieve(query, index, chunks, k=12):
    # Convert question to embedding
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding)

    # Search in FAISS
    distances, indices = index.search(query_embedding, k)

    # Get top chunks
    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [14]:
query = "What is classical conditioning?"

results = retrieve(query, index, chunks)

for r in results:
    print("Section:", r["section"])
    print("Page:", r["page_number"])
    print("Text Preview:", r["text"][:200])
    print("---------------")

Section: 6.2 In operant conditioning
Page: 195
Text Preview: and his famous dogs. Pavlov (1849–1936), a Russian scientist, performed extensive research on dogs and is best known for his experiments in classical conditioning (Figure 6.3). As we discussed briefly
---------------
Section: 6.6 Kate holds a southern stingray at Stingray City in the Cayman Islands
Page: 200
Text Preview: General Processes in Classical Conditioning Now that you know how classical conditioning works and have seen several examples, let’s take a look at some of the general processes involved. In classical
---------------
Section: 6.1
Psychologist B
Page: 205
Text Preview: Classical and Operant Conditioning Compared Classical Conditioning Operant Conditioning Conditioning approach An unconditioned stimulus (such as food) is paired with a neutral stimulus (such as a bell
---------------
Section: 6.4 Before conditioning
Page: 197
Text Preview: FIGURE 6.4 Before conditioning, an unconditioned stimulus (food) produce

In [15]:
def build_context(retrieved_chunks):
    context_parts = []
    sections = set()
    pages = set()

    for chunk in retrieved_chunks:
        context_parts.append(chunk["text"])
        sections.add(chunk["section"])
        pages.add(chunk["page_number"])

    context = "\n\n".join(context_parts)

    return context, list(sections), list(pages)

In [16]:
context, sections, pages = build_context(results)

print("Sections:", sections)
print("Pages:", pages)

Sections: ['15.13 Different regions of the brain may be associated with different psychological disorders', '6.1 What Is Learning', '6.1\nPsychologist B', '6.7 This is the curve of acquisition', '16.9 This is the famous couch in Freud', '16.11 Exposure therapy seeks to change the response to a conditioned stimulus', '6.6 Kate holds a southern stingray at Stingray City in the Cayman Islands', '6.2 In operant conditioning', '6.4 Before conditioning']
Pages: [194, 195, 197, 200, 202, 205, 622, 621, 569]


In [17]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_BASE_URL = os.getenv("NVIDIA_BASE_URL")

In [50]:
from openai import OpenAI

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY
)

def generate_answer(prompt):
    response = client.chat.completions.create(
        model="meta/llama3-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=300
    )
    
    return response.choices[0].message.content

In [52]:
def build_prompt(question, context):
    return f"""
You are answering a psychology textbook question.

Use ONLY the provided textbook context.

You may combine information from multiple parts of the context to form the answer.
Summarize clearly using the textbook content.

Respond with:
Not found in the provided textbook.
ONLY if there is absolutely no relevant information in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [53]:
with open("data/queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print("Total queries:", len(queries))

Total queries: 50


In [54]:
results = []

for q in tqdm(queries):
    query_id = q["query_id"]
    question = q["question"]

    try:
        # 1 Retrieve with higher k
        retrieved = retrieve(question, index, chunks, k=12)

        # 2 Build context
        context, sections, pages = build_context(retrieved)

        # 3 DO NOT aggressively truncate
        # Only truncate if extremely large
        if len(context) > 10000:
            context = context[:10000]

        # 4 Build prompt
        prompt = build_prompt(question, context)

        # 5 Generate answer
        answer = generate_answer(prompt)

        # 6 Safety check: if model returns empty string
        if not answer or answer.strip() == "":
            answer = "Not found in the provided textbook."

    except Exception as e:
        print(f"Error at ID {query_id}:", e)
        context = ""
        answer = "Not found in the provided textbook."
        sections = []
        pages = []

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer.strip(),
        "references": json.dumps({
            "sections": sections,
            "pages": pages
        })
    })

print("Total results:", len(results))

100%|██████████| 50/50 [06:05<00:00,  7.31s/it]

Total results: 50


In [56]:
df = pd.DataFrame(results)
df = df[["ID", "context", "answer", "references"]]

print("Total rows:", len(df))

df.to_csv("submission.csv", index=False)

print("submission.csv generated successfully!")

Total rows: 50
submission.csv generated successfully!


In [55]:
test_prompt = "What are applications of psychology in education?"
print(generate_answer(test_prompt))

Psychology plays a crucial role in education, and its applications are numerous and diverse. Here are some of the key applications of psychology in education:

1. **Learning Theories**: Understanding how students learn, retain, and apply information is essential in education. Psychology helps educators develop effective learning strategies, such as spaced repetition, active learning, and retrieval practice.
2. **Instructional Design**: Psychology informs the design of instructional materials, including textbooks, online courses, and educational software. By understanding how students process information, educators can create engaging and effective learning experiences.
3. **Classroom Management**: Psychology helps teachers manage classroom behavior, create a positive learning environment, and promote social skills, such as cooperation and conflict resolution.
4. **Assessment and Evaluation**: Psychological principles guide the development of assessments and evaluations that accurately 

In [24]:
test_prompt = "What are the basic parts of a neuron?"
print(generate_answer(test_prompt))

The basic parts of a neuron, also known as a nerve cell, are:

1. **Dendrites**: These are branching extensions of the neuron that receive signals from other neurons. They are like antennae, picking up signals from other cells.
2. **Cell Body** (Soma): This is the central part of the neuron where the cell's genetic material (DNA) is located. It's like the neuron's headquarters, where all the cell's activities are controlled.
3. **Axon**: This is a long, slender extension of the neuron that carries signals away from the cell body to other neurons, muscles, or glands. It's like a highway, transmitting signals to other cells.
4. **Myelin Sheath**: This is a fatty, insulating layer that surrounds the axon, helping to speed up the transmission of signals. It's like a protective coating, allowing the signal to travel faster.
5. **Terminal Buttons** (Synaptic Terminals): These are the ends of the axon, where the neuron releases chemical messengers called neurotransmitters into the synapse (th

In [24]:
retrieved = retrieve("What is the difference between mood disorders and personality disorders?", index, chunks, k=12)

for r in retrieved:
    print("SECTION:", r["section"])
    print("PAGE:", r["page_number"])
    print(r["text"][:400])
    print("------")

SECTION: 15.15 Mood disorders are characterized by massive disruptions in mood
PAGE: 573
depression, but also mania and elation (Rothschild, 1999). All of us experience fluctuations in our moods and emotional states, and often these fluctuations are caused by events in our lives. We become elated if our favorite team wins the World Series and dejected if a romantic relationship ends or if we lose our job. At times, we feel fantastic or miserable for no clear reason. People with mood d
------
SECTION: 15.7 Mood and Related Disorders
LEARNING OBJECTIVES
By the end of this section
PAGE: 572
preventing both a change in the nature of the memory and a change in the problematic appraisals. 15.7 Mood and Related Disorders LEARNING OBJECTIVES By the end of this section, you will be able to: • Distinguish normal states of sadness and euphoria from states of depression and mania • Describe the symptoms of major depressive disorder and bipolar disorder • Understand the differences between maj
----